In [2]:
import json
import time
from getpass import getpass
from typing import Any

import paramiko


def connect_pi(
    host: str,
    username: str,
    password: str | None = None,
    key_filename: str | None = None,
    port: int = 22,
    timeout: int = 10,
    prompt_for_password: bool = True,
) -> paramiko.SSHClient:
    """Open and return an SSH connection to the Raspberry Pi."""
    if password is None and key_filename is None and prompt_for_password:
        password = getpass(f"Password for {username}@{host}: ")

    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    try:
        client.connect(
            hostname=host,
            port=port,
            username=username,
            password=password,
            key_filename=key_filename,
            timeout=timeout,
            allow_agent=True,
            look_for_keys=True,
        )
    except paramiko.AuthenticationException as exc:
        raise RuntimeError(
            "Authentication failed. Set PI_PASSWORD or PI_KEY_FILE correctly."
        ) from exc
    except paramiko.SSHException as exc:
        raise RuntimeError(
            "SSH negotiation/auth setup failed. Check that SSH is enabled on the Pi and credentials are valid."
        ) from exc

    return client


def _run_remote_command(client: paramiko.SSHClient, command: str) -> tuple[str, str, int]:
    """Run a command on the Pi and return stdout, stderr, and exit status."""
    stdin, stdout, stderr = client.exec_command(command)
    _ = stdin

    raw_output = stdout.read().decode("utf-8").strip()
    raw_error = stderr.read().decode("utf-8").strip()
    exit_status = stdout.channel.recv_exit_status()
    return raw_output, raw_error, exit_status


def _run_remote_json_command(client: paramiko.SSHClient, command: str) -> dict[str, Any]:
    """Run a Pi command that returns JSON and decode it."""
    raw_output, raw_error, exit_status = _run_remote_command(client=client, command=command)

    if exit_status != 0:
        raise RuntimeError(
            f"Pi command failed with exit status {exit_status}. STDERR: {raw_error}"
        )
    if raw_error:
        raise RuntimeError(f"Pi returned an error: {raw_error}")
    if not raw_output:
        raise RuntimeError("Pi command returned no output.")

    try:
        return json.loads(raw_output)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Expected JSON from Pi, got: {raw_output}") from exc


def build_remote_light_command(light_sensor_port: int = 0) -> str:
    """Build a Python command that runs on the Pi and returns JSON with a light reading."""
    return (
        "python3 -c \""
        "import json,time,grovepi;"
        f"value=int(grovepi.analogRead({light_sensor_port}));"
        f"print(json.dumps({{'sensor':'light','port':{light_sensor_port},'value':value,'timestamp':time.time()}}))"
        "\""
    )


def get_light_reading(client: paramiko.SSHClient, light_sensor_port: int = 0) -> dict[str, Any]:
    """Read one light sensor value from the Raspberry Pi over an existing SSH connection."""
    command = build_remote_light_command(light_sensor_port=light_sensor_port)
    return _run_remote_json_command(client=client, command=command)


def stream_light_readings(
    client: paramiko.SSHClient,
    sample_count: int = 10,
    interval_seconds: float = 1.0,
    light_sensor_port: int = 0,
) -> list[dict[str, Any]]:
    """Collect multiple light readings from the Raspberry Pi at a fixed interval."""
    readings: list[dict[str, Any]] = []
    for _ in range(sample_count):
        readings.append(get_light_reading(client=client, light_sensor_port=light_sensor_port))
        time.sleep(interval_seconds)
    return readings


def build_remote_temperature_command(dht_sensor_port: int = 5, dht_sensor_type: int = 0) -> str:
    """Build a command that reads DHT temperature/humidity on the Pi and returns JSON."""
    return (
        "python3 -c \""
        "import json,time,grovepi,math;"
        f"temp_c,humidity=grovepi.dht({dht_sensor_port},{dht_sensor_type});"
        "temp_c=float(temp_c) if temp_c is not None else float('nan');"
        "humidity=float(humidity) if humidity is not None else float('nan');"
        "ok=not (math.isnan(temp_c) or math.isnan(humidity));"
        "temp_f=(temp_c*9/5+32) if ok else float('nan');"
        f"print(json.dumps({{'sensor':'dht','port':{dht_sensor_port},'sensor_type':{dht_sensor_type},'temperature_c':temp_c,'temperature_f':temp_f,'humidity':humidity,'ok':ok,'timestamp':time.time()}}))"
        "\""
    )


def get_temperature_reading(
    client: paramiko.SSHClient,
    dht_sensor_port: int = 5,
    dht_sensor_type: int = 0,
) -> dict[str, Any]:
    """Read one DHT temperature/humidity value from the Raspberry Pi."""
    command = build_remote_temperature_command(
        dht_sensor_port=dht_sensor_port,
        dht_sensor_type=dht_sensor_type,
    )
    data = _run_remote_json_command(client=client, command=command)
    if not data.get("ok", False):
        raise RuntimeError(
            "DHT read returned invalid data. Try reading again in a second."
        )
    return data


def play_audio_on_pi(
    client: paramiko.SSHClient,
    audio_file_path: str,
    blocking: bool = True,
) -> dict[str, Any]:
    """Play an audio file on Pi speakers using available player tools."""
    mode = "blocking" if blocking else "background"
    command = (
        "python3 -c \""
        "import json,os,shutil,subprocess;"
        f"audio_path={audio_file_path!r};"
        f"mode={mode!r};"
        "exists=os.path.isfile(audio_path);"
        "players=['aplay','mpg123','omxplayer'];"
        "player=next((p for p in players if shutil.which(p)),None);"
        "result={'audio_path':audio_path,'exists':exists,'player':player,'mode':mode};"
        "if (not exists) or (player is None):"
        " print(json.dumps(result));"
        " raise SystemExit(0);"
        "cmd=[player,audio_path];"
        "if mode=='background':"
        " subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL);"
        " result['started']=True;"
        " result['returncode']=None;"
        "else:"
        " proc=subprocess.run(cmd, capture_output=True, text=True);"
        " result['started']=(proc.returncode==0);"
        " result['returncode']=proc.returncode;"
        " result['stderr']=proc.stderr.strip();"
        "print(json.dumps(result))"
        "\""
    )

    result = _run_remote_json_command(client=client, command=command)
    if not result.get("exists", False):
        raise FileNotFoundError(
            f"Audio file not found on Pi: {audio_file_path}"
        )
    if result.get("player") is None:
        raise RuntimeError(
            "No supported player found on Pi. Install aplay, mpg123, or omxplayer."
        )
    if blocking and not result.get("started", False):
        raise RuntimeError(
            f"Audio playback failed. Player stderr: {result.get('stderr', '')}"
        )
    return result


def close_pi_connection(client: paramiko.SSHClient) -> None:
    """Close an open SSH connection to the Raspberry Pi."""
    client.close()

## connect to raspi

In [14]:
# Example usage from your PC
PI_HOST = "10.19.0.68"
PI_USERNAME = "pi"  # change if your Pi username is different
PI_PASSWORD = 'W1n0nA'    # optionally set directly, otherwise you will be prompted
PI_KEY_FILE = None    # optional: e.g., r"C:\\Users\\you\\.ssh\\id_rsa"

# if PI_PASSWORD is None and PI_KEY_FILE is None:
#     PI_PASSWORD = getpass(f"Password for {PI_USERNAME}@{PI_HOST}: ")

client = connect_pi(
    host=PI_HOST,
    username=PI_USERNAME,
    password=PI_PASSWORD,
    key_filename=PI_KEY_FILE,
    prompt_for_password=False,
)



# Optional: play an audio file that already exists on the Pi
# audio_result = play_audio_on_pi(
#     client=client,
#     audio_file_path="/home/pi/test.wav",
#     blocking=True,
# )
# audio_result

#close_pi_connection(client)

In [15]:
get_light_reading(client=client, light_sensor_port=0) # find light reading

{'sensor': 'light', 'port': 0, 'value': 776, 'timestamp': 1776180748.2459316}

In [19]:
get_temperature_reading(
    client=client,
    dht_sensor_port=5,
    dht_sensor_type=0,
) # find temperature reading

{'sensor': 'dht',
 'port': 5,
 'sensor_type': 0,
 'temperature_c': 24.0,
 'temperature_f': 75.2,
 'humidity': 47.0,
 'ok': True,
 'timestamp': 1776181819.9275906}